# Домашнее задание 6. Автоматизация расчетов с помощью Apache AirFlow

## Шаг 1. Создать Airflow DAG, который будет ежедневно выполнять следующие действия:
загружать данные из файлов с локальной директории в соответствующие таблицы (параллельные task-и):
о клиентах — customer;
о продуктах — product;
о заказах — orders;
связывающие заказы и продукты — order_items.

In [ ]:
from datetime import datetime, timedelta
import logging
import os
import pandas as pd
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.postgres.hooks.postgres import PostgresHook
from airflow.models import Variable
from airflow.exceptions import AirflowException

# Настройка логгера
logger = logging.getLogger(__name__)

# Callback функции для обработки событий
def on_failure_callback(context):
    """Callback функция при неудачном выполнении задачи"""

    error_message = f"""
    ЗАДАЧА ПРОВАЛЕНА!
    """
    logger.error(error_message)

def on_success_callback(context):
    """Callback функция при успешном выполнении задачи"""
    success_message = f"""
    ЗАДАЧА ВЫПОЛНЕНА УСПЕШНО!
    """
    logger.info(success_message)

# Аргументы DAG с callback функциями
default_args = {
    'owner': Variable.get('data_load_owner', default_var='data_engineer'),
    'depends_on_past': Variable.get('data_load_depends_on_past', default_var=False),
    'start_date': datetime(2025, 1, 1),
    'retries': 3,
    'retry_delay': timedelta(minutes=5),
    'on_failure_callback': on_failure_callback,
    'on_success_callback': on_success_callback
}

def load_csv_to_postgres(table_name, csv_file_path):
    """Загрузка данных из CSV файла в таблицу PostgreSQL"""
    try:
        # Проверка существования файла
        if not os.path.exists(csv_file_path):
            raise AirflowException(f"Файл не найден: {csv_file_path}")

        logger.info(f"Чтение CSV файла: {csv_file_path}")
        df = pd.read_csv(csv_file_path)

        logger.info(f"Загрузка {len(df)} строк в таблицу {table_name}")

        # Подключение к PostgreSQL
        pg_hook = PostgresHook(postgres_conn_id='postgres_default')
        conn = pg_hook.get_conn()
        cursor = conn.cursor()

        # Вставка данных
        for _, row in df.iterrows():
            columns = ', '.join(row.index)
            values = ', '.join([f"'{str(val)}'" if pd.notna(val) else 'NULL' for val in row.values])
            query = f"INSERT INTO {table_name} ({columns}) VALUES ({values});"
            cursor.execute(query)

        conn.commit()
        cursor.close()
        conn.close()

        logger.info(f"Данные успешно загружены в таблицу {table_name}")
        return True

    except Exception as e:
        logger.error(f"Ошибка при загрузке данных в таблицу {table_name}: {str(e)}")
        raise AirflowException(f"Data load failed for {table_name}: {str(e)}")

with DAG(
    dag_id='daily_data_load_from_csv',
    default_args=default_args,
    description='Ежедневная загрузка данных из CSV файлов в PostgreSQL',
    schedule_interval='0 6 * * *',  # Запуск в 06:00 ежедневно
    catchup=False,
    tags=['data_load', 'etl', 'production'],
    max_active_runs=1,
) as dag:

    # Определение путей к CSV файлам
    csv_paths = {
        'customer': '/path/customers.csv',
        'product': '/path/products.csv',
        'orders': '/path/orders.csv',
        'order_items': '/path/order_items.csv'
    }

    # Задача проверки подключения к БД
    check_db_connection_task = PythonOperator(
        task_id='check_database_connection',
        python_callable=lambda: PostgresHook(postgres_conn_id='postgres_default').get_conn()
    )

    # Создание задач для параллельной загрузки данных
    load_tasks = {}
    for table, csv_path in csv_paths.items():
        load_tasks[table] = PythonOperator(
            task_id=f'load_{table}_data',
            python_callable=load_csv_to_postgres,
            op_kwargs={
                'table_name': table,
                'csv_file_path': csv_path
            },
            provide_context=True
        )

    # Определяем порядок выполнения задач
    check_db_connection_task >> [load_tasks['customer'], load_tasks['product'], load_tasks['orders'], load_tasks['order_items']]


## Шаг 2. Далее выполнить следующие запросы, записав ответы в отдельные файлы (параллельные task):
Найти имена и фамилии клиентов с ТОП-3 минимальной и ТОП-3 максимальной суммой транзакций за весь период (учесть клиентов, у которых нет заказов).
Найти ТОП-5 клиентов (по общему доходу) в каждом сегменте благосостояния (wealth_segment). Вывести имя, фамилию, сегмент и общий доход. Если в сегменте менее 5 клиентов, вывести всех.

In [ ]:

def run_analysis_query(query, output_file):
    """Выполняет SQL-запрос и сохраняет результат в CSV"""
    try:
        pg_hook = PostgresHook(postgres_conn_id='postgres_default')
        df = pg_hook.get_pandas_df(query)

        df.to_csv(output_file, index=False)
        logger.info(f"Результат сохранён в {output_file}")
        return True

    except Exception as e:
        logger.error(f"Ошибка при выполнении запроса: {str(e)}")
        raise AirflowException(f"Query failed: {str(e)}")

with DAG(
    dag_id='daily_data_load_and_analysis',
    default_args=default_args,
    description='Ежедневная загрузка данных и анализ',
    schedule_interval='0 6 * * *',
    catchup=False,
    tags=['data_load', 'etl', 'analysis'],
    max_active_runs=1,
) as dag:

    csv_paths = {
        'customer': '/path/customers.csv',
        'product': '/path/products.csv',
        'orders': '/path/orders.csv',
        'order_items': '/path/order_items.csv'
    }

    output_dir = '/path/output/analysis'  # Директория для выходных файлов

    # Задача проверки подключения
    check_db_connection_task = PythonOperator(
        task_id='check_database_connection',
        python_callable=lambda: PostgresHook(postgres_conn_id='postgres_default').get_conn()
    )

    # Задачи загрузки данных (параллельные)
    load_tasks = {}
    for table, csv_path in csv_paths.items():
        load_tasks[table] = PythonOperator(
            task_id=f'load_{table}_data',
            python_callable=load_csv_to_postgres,
            op_kwargs={'table_name': table, 'csv_file_path': csv_path},
            provide_context=True
        )

    # Анализ 1: ТОП-3 мин/макс суммы транзакций (с учётом клиентов без заказов)
    top_transactions_query = """
    WITH customer_total AS (
        SELECT
            c.customer_id,
            c.first_name,
            c.last_name,
            COALESCE(SUM(oi.quantity * oi.item_list_price_at_sale), 0) AS total_spent
        FROM customer c
        LEFT JOIN orders o ON c.customer_id = o.customer_id
        LEFT JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY c.customer_id, c.first_name, c.last_name
    )
    SELECT first_name, last_name, total_spent
    FROM (
        (SELECT first_name, last_name, total_spent FROM customer_total ORDER BY total_spent ASC LIMIT 3)
        UNION ALL
        (SELECT first_name, last_name, total_spent FROM customer_total ORDER BY total_spent DESC LIMIT 3)
    ) AS top_bottom
    ORDER BY total_spent;
    """

    analysis_task_1 = PythonOperator(
        task_id='analyze_top_bottom_customers',
        python_callable=run_analysis_query,
        op_kwargs={
            'query': top_transactions_query,
            'output_file': os.path.join(output_dir, 'top_bottom_customers.csv')
        }
    )

    # Анализ 2: ТОП-5 клиентов по доходу в каждом сегменте благосостояния
    top_by_segment_query = """
    WITH customer_wealth AS (
        SELECT
            c.customer_id,
            c.first_name,
            c.last_name,
            c.wealth_segment,
            COALESCE(SUM(oi.quantity * oi.item_list_price_at_sale), 0) AS total_revenue
        FROM customer c
        LEFT JOIN orders o ON c.customer_id = o.customer_id
        LEFT JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY c.customer_id, c.first_name, c.last_name, c.wealth_segment
    ),
    ranked_customers AS (
        SELECT
            first_name,
            last_name,
            wealth_segment,
            total_revenue,
            ROW_NUMBER() OVER (PARTITION BY wealth_segment ORDER BY total_revenue DESC) AS rn
        FROM customer_wealth
    )
    SELECT first_name, last_name, wealth_segment, total_revenue
    FROM ranked_customers
    WHERE rn <= 5
    ORDER BY wealth_segment, total_revenue DESC;
    """

    analysis_task_2 = PythonOperator(
        task_id='analyze_top_customers_by_segment',
        python_callable=run_analysis_query,
        op_kwargs={

## Шаг 3. Проверить, что запросы из пункта выше не вернули нулевое количество строк. В случае непрохождения проверки выводит сообщение. (корректировка задания на семинаре)


In [ ]:
# Настройка логгера
logger = logging.getLogger(__name__)

# Callback-функции
def on_failure_callback(context):
    error_message = f"""
    ПРОВЕРКА ПРОВАЛЕНА!
    """
    logger.error(error_message)

def on_success_callback(context):
    success_message = f"""
    ПРОВЕРКА ВЫПОЛНЕНА УСПЕШНО!
    """
    logger.info(success_message)

# Аргументы DAG
default_args = {
    'owner': Variable.get('validator_owner', default_var='data_engineer'),
    'depends_on_past': False,
    'start_date': datetime(2026, 1, 1),
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
    'on_failure_callback': on_failure_callback,
    'on_success_callback': on_success_callback,
}

def validate_analysis_results(**kwargs):
    """
    Проверяет, что аналитические CSV-файлы существуют и содержат данные.
    Поднимает исключение, если файл отсутствует или пуст.
    """
    # Пути к проверяемым файлам
    output_dir = '/path/output/analysis'
    files_to_check = [
        os.path.join(output_dir, 'top_bottom_customers.csv'),
        os.path.join(output_dir, 'top_customers_by_wealth_segment.csv')
    ]

    all_passed = True
    for file_path in files_to_check:
        try:
            if not os.path.exists(file_path):
                logger.error(f"Файл не найден: {file_path}")
                all_passed = False
                continue

            df = pd.read_csv(file_path)
            if df.empty:
                logger.error(f"Файл {file_path} пуст! Результат анализа отсутствует.")
                all_passed = False
            else:
                logger.info(f"Файл {file_path} содержит {len(df)} строк. Проверка пройдена.")


        except Exception as e:
            logger.error(f"Ошибка при проверке файла {file_path}: {str(e)}")
            all_passed = False

    if not all_passed:
        raise ValueError("Одна или несколько проверок не прошли.")
    return True

with DAG(
    dag_id='validation_analysis_results',
    default_args=default_args,
    description='Проверка результатов аналитических запросов',
    schedule_interval='0 6 * * *',  # Запуск в 06:00 (после основного DAG)
    catchup=False,
    tags=['validation', 'analysis', 'monitoring'],
    max_active_runs=1,
) as dag:


    validation_task = PythonOperator(
        task_id='check_analysis_output_files',
        python_callable=validateanalysis_results,
        provide_context=True,
    )

    validation_task


# Шаг 4. Вывести сообщение об успешном или неуспешном выполнении DAG.

In [ ]:
from airflow.models import DagRun, TaskInstance
from airflow.utils.state import State


# Настройка логгера
logger = logging.getLogger(__name__)

# Callback-функции
def on_failure_callback(context):
    error_message = f"""
    ИТОГ: НЕУСПЕШНО
    DAG: {context['dag'].dag_id}
    TASK: {context['task'].task_id}
    EXECUTION_DATE: {context['execution_date']}
    REASON: Одна или несколько зависимостей завершились с ошибкой.
    """
    logger.error(error_message)

def on_success_callback(context):
    success_message = f"""
    ИТОГ: УСПЕШНО
    DAG: {context['dag'].dag_id}
    TASK: {context['task'].task_id}
    EXECUTION_DATE: {context['execution_date']}
    Все зависимости выполнены успешно.
    """
    logger.info(success_message)

# Аргументы DAG
default_args = {
    'owner': Variable.get('status_monitor_owner', default_var='ops'),
    'depends_on_past': False,
    'start_date': datetime(2026, 1, 1),
    'retries': 0,
    'retry_delay': timedelta(minutes=5),
    'on_failure_callback': on_failure_callback,
    'on_success_callback': on_success_callback,
}

def check_dag_statuses(**kwargs):
    """
    Проверяет статус выполнения указанных DAG за текущую дату.
    Считает успех, если все задачи в каждом DAG завершены успешно.
    """
    # Имена DAG, которые нужно проверить
    target_dags = [
        'daily_data_load_analysis_validation',  # Основной DAG загрузки и анализа
        'validation_analysis_results'          # DAG валидации результатов
    ]

    execution_date = kwargs['execution_date'].date()
    all_successful = True

    for dag_id in target_dags:
        # Ищем последний запуск DAG за текущую дату
        dag_runs = DagRun.find(dag_id=dag_id, execution_date=execution_date)
        if not dag_runs:
            logger.warning(f"DAG {dag_id} не запускался сегодня ({execution_date}).")
            all_successful = False
            continue

        # Берём последний запуск (по дате)
        latest_run = max(dag_runs, key=lambda x: x.execution_date)

        if latest_run.state != State.SUCCESS:
            logger.error(f"DAG {dag_id} завершился со статусом: {latest_run.state}")
            all_successful = False
        else:
            # Дополнительно проверяем статус всех задач в DAG
            session = kwargs['dag'].get_session()
            task_instances = session.query(TaskInstance).filter(
                TaskInstance.dag_id == dag_id,
                TaskInstance.execution_date == latest_run.execution_date
            ).all()

            for ti in task_instances:
                if ti.state != State.SUCCESS:
                    logger.error(
                        f"Задача {ti.task_id} в DAG {dag_id} завершилась со статусом: {ti.state}"
                    )
                    all_successful = False

    if all_successful:
        logger.info("Все целевые DAG выполнены успешно.")
        return True
    else:
        raise AirflowException("Один или несколько DAG завершились с ошибкой или не выполнялись.")

with DAG(
    dag_id='monitor_dag_status',
    default_args=default_args,
    description='Мониторинг статуса выполнения ключевых DAG',
    schedule_interval='0 7 * * *',  # Запуск в 07:00, после всех предыдущих DAG
    catchup=False,
    tags=['monitoring', 'status', 'operations'],
    max_active_runs=1,
) as dag:

    status_check_task = PythonOperator(
        task_id='check_target_dag_statuses',
        python_callable=check_dag_statuses,
        provide_context=True,
    )

    status_check_task
